# **PROYEK UAS**

## Nama Proyek : **Melakukan klasifikasi time series pada dataset DucksAndGeese**

### A. **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*)

**1. *Business Understanding* (Pemahaman Bisnis)**

**1.1 Latar Belakang**

Identifikasi spesies burung merupakan bagian penting dalam bidang ekologi dan konservasi, khususnya untuk keperluan monitoring populasi dan pengamatan biodiversitas. Namun, identifikasi burung secara visual sering kali sulit dilakukan karena keterbatasan kondisi lapangan, seperti burung yang tersembunyi di vegetasi, aktif pada waktu tertentu, atau berada di lokasi yang sulit dijangkau oleh pengamat.

Salah satu pendekatan alternatif yang banyak digunakan adalah bioakustik, yaitu identifikasi spesies berdasarkan karakteristik suara. Dataset **DucksAndGeese** yang berasal dari *UEA Time Series Archive* menyediakan sinyal audio berbentuk *time series* yang merepresentasikan lima spesies burung air, yang terdiri dari dua spesies bebek (*duck*) dan tiga spesies angsa (*geese*).

Permasalahan utama pada dataset ini adalah bagaimana mengklasifikasikan sinyal *audio time series* tersebut ke dalam kategori spesies burung yang benar, mengingat adanya variasi pola suara, *noise* lingkungan, serta kemiripan karakteristik frekuensi antar spesies.

**1.2 Tujuan Proyek**

Membangun dan mengevaluasi model *time series classification* berbasis *machine learning* yang mampu mengklasifikasikan sinyal audio burung ke dalam lima kelas spesies berikut:
- Black-bellied Whistling Duck
- Canadian Goose
- Greylag Goose
- Pink-footed Goose
- White-faced Whistling Duck
Permasalahan ini termasuk dalam klasifikasi multikelas (*multiclass classification*) dengan lima label kelas diskrit, di mana setiap sinyal audio direpresentasikan sebagai satu *instance data*.

**1.3 Manfaat**

Hasil dari klasifikasi sinyal audio pada penelitian ini diharapkan dapat dimanfaatkan sebagai dasar pengembangan **sistem identifikasi bioakustik otomatis**, dengan manfaat sebagai berikut:
- Mendukung monitoring fauna secara otomatis tanpa ketergantungan pada observasi visual.
- Membantu pengamatan dan pemetaan biodiversitas burung di alam liar.
- Mengurangi ketergantungan pada identifikasi manual oleh manusia, yang cenderung memerlukan waktu, tenaga, dan keahlian khusus.
- Menjadi dasar pengembangan aplikasi berbasis audio untuk keperluan ekologi dan konservasi satwa liar.

**1.4 Kriteria kesuksesan**

Model dikatakan berhasil jika mencapai:
- **Accuracy ≥ 85%** untuk *baseline*.
- **F1-score** per kelas **≥ 80%**, untuk memastikan performa yang seimbang pada seluruh spesies.
- Aplikasi Streamlit berjalan lancar dan mampu menerima input serta menampilkan prediksi dan *confidence score*.

**2. *Data Understanding* (Pemahaman Data)**

**2.1 Gambaran Umum Dataset**

Dataset **DucksAndGeese** berasal dari *UEA Time Series Archive* dan berisi sinyal *time series audio univariate* yang merepresentasikan suara dari 5 spesies burung air, terdiri atas 2 jenis bebek (*duck*) dan 3 jenis angsa (*geese*).

Dataset ini digunakan untuk permasalahan klasifikasi time series multikelas, di mana setiap *instance* berupa satu rekaman audio mono yang telah diproses menjadi panjang sinyal yang seragam. Dataset ini merupakan versi *univariate* dari dataset **DuckDuckGeese** yang bersifat *multivariate*.

Setiap *instance* direpresentasikan sebagai urutan nilai amplitudo audio (*raw waveform*) yang kemudian dipasangkan dengan label kelas spesies.

Berdasarkan dokumentasi *UEA Time Series Archive* dan file **.ts**, spesifikasi dataset adalah sebagai berikut:

Tabel Spesifikasi Dataset
| Properti                             | Nilai                                  |
|--------------------------------------|----------------------------------------|
| *Train Size*                         | 50                                     |
| *Test Size*                          | 50                                     |
| *Total Instances*                    | 100                                    |
| *Time Series Length*                 | 236,784 titik data per *instance*      |
| *Number of Classes*                  | 5                                      |
| *Dimensions*                         | 1 (*audio waveform univariate*)        |
| *Datatype*                           | Float (*amplitudo audio*)              |
| *Data format*                        | AUDIO (*raw waveform*)                 |
| *Sample rate* setelah *preprocessing*| 44,100 Hz                              |
| *Data Split*                         | TRAIN dan TEST                         |
| Sumber                               | www.xenocanto.com                      |

Dengan asumsi *sample rate* 44.100 Hz, setiap rekaman memiliki durasi sekitar:
236.784 / 44.100 ≈ 5,37 detik

Tabel Kelas dan Distribusi Label
| Label | Spesies                          | Jumlah |
|-------|----------------------------------|--------|
| 0     | Black-bellied Whistling Duck     | 20     |
| 1     | Canadian Goose                   | 20     |
| 2     | Greylag Goose                    | 20     |
| 3     | Pink-footed Goose                | 20     |
| 4     | White-faced Whistling Duck       | 20     |

Karena jumlah data tiap kelas sama, dataset tidak mengalami *class imbalance*, sehingga tidak diperlukan teknik *oversampling* atau *undersampling* pada tahap *preprocessing*.

**2.2 Asal-usul dan Proses Pembentukan Data**

Berdasarkan dokumentasi **UEA Archive**:
- Audio berasal dari rekaman lapangan pada platform **Xeno-Canto**, sebuah repositori suara burung dan komunitas *ornithology*.
- Rekaman memiliki variasi *sample rate* dan durasi.
- Seluruh audio kemudian:
    1. Diseragamkan *sample rate*-nya yang awalnya berbeda-beda (misalnya 48kHz, 96kHz, 22kHz) yaitu *downsampling* ke 44.100 Hz.
    2. Dipangkas (*truncation*) ke panjang 236.784 data point, yang merupakan panjang terpendek dari seluruh rekaman karena panjang aslinya bervariasi.
- Rekaman suara berasal dari kategori kualitas A (noise rendah) atau B (noise sedang), berarti suara cukup bersih.
- Dataset telah disediakan dalam bentuk **TRAIN** (50) dan **TEST** (50), sehingga evaluasi dapat dilakukan langsung pada *test set* tanpa kebocoran data. Artinya, setiap *instance* merepresentasikan potongan audio berdurasi sama, sehingga model fokus pada pola temporal suara, bukan perbedaan panjang sinyal. Untuk pengembangan, split ulang TRAIN menjadi *train* / *validation* (80/20) untuk *tuning hyperparameter* dan memilih model sebelum uji pada TEST final.

**2.3 Struktur Data di Dalam File .ts**

Setelah membuka file TRAIN dan TEST:
- Setiap baris merepresentasikan 1 *instance*
- Format: TS format (UEA standard)
- Setiap baris berisi:
    v1, v2, v3, ..., v236784 : classLabel
- classLabel menunjukkan kelas spesies (0–4)
- Nilai v merupakan nilai amplitudo audio bertipe float
    ex : 0.0012, -0.0031, 0.0042, ... , -0.0008 : 3 (Artinya data tersebut adalah Pink-footed Goose)

**2.4 Karakteristik Data**

**2.4.1 *Univariate Long Sequence***

- Panjang data sangat besar (236k titik)
- Membutuhkan:
    - optimasi memori
    - kemungkinan segmentasi / ekstraksi fitur
- Pendekatan:
    - CNN-*based Time Series Classification*
    - *Distance-based method* (DTW)
    - *Feature-based ML* (untuk ekstraksi)

**2.4.2 *Balanced Classes***

Semua kelas masing-masing 20 sampel, sehingga:
- Tidak perlu balancing
- Setiap kelas memiliki jumlah instance yang sama
- Evaluasi model lebih stabil
- Accuracy dan Macro-F1 relevan digunakan

**2.4.3 *High Variability***

Karena suara burung:
- Rekaman berasal dari lingkungan alami, sehingga :
    - Banyak noise lingkungan
    - Bentuk gelombang sangat bervariasi
    - *High-frequency chirp*/*whistle patterns*

**2.4.4 *Data Challenges***

- Ukuran input besar (*memory heavy*)
- Harus distandarisasi (z-score)
- Perlu *filtering* (*bandpass* 300–8000 Hz)
- Potensial *cropping* / *feature extraction* (MFCC)

**2.5 EDA (*Exploratory Data Analysis*)**

**2.5.1 Struktur Dataset**

| File                   | Isi                              |
|------------------------|----------------------------------|
| DucksAndGeese_TRAIN.ts | 50 instance, univariate, labeled |
| DucksAndGeese_TEST.ts  | 50 instance, univariate, labeled |

**2.5.2 Basic Stats**

| Statistik              | Nilai                      |
|------------------------|----------------------------|
| Total sample           | 100                        |
| Panjang time series    | 236,784 *point per sample* |
| Channel                | 1 (mono)                   |
| Durasi estimasi        | 5.37 detik per sample      |

**2.5.3 Distribusi Label**

Semua kelas memiliki 20 sampel ***balanced***.
| Label | Nama Spesies                      | Jumlah |
|-------|-----------------------------------|--------|
| 0     | *Black-bellied Whistling Duck*    | 20     |
| 1     | *Canadian Goose*                  | 20     |
| 2     | *Greylag Goose*                   | 20     |
| 3     | *Pink-footed Goose*               | 20     |
| 4     | *White-faced Whistling Duck*      | 20     |

**2.5.4 *Visualisasi Waveform***

Tujuan EDA pada *audio time series*:
- melihat noise
- melihat pola amplitudo
- melihat perbedaan antar spesies

Insight yang diharapkan:
1. *Canadian Goose* = nada lebih rendah (gelombang besar & lambat)
2. *Whistling Duck* = suara nyaring (gelombang cepat & rapat)
3. *Greylag Goose* = tone cenderung clustering

**2.5.5 Analisis Statistik per Kelas**

Hitung:
- mean amplitude
- standard deviation
- zero crossing count
- RMS energy
ex :    | Kelas | Mean | Std  | ZeroCrossRate   | RMS |
        |-------|------|------|---------------- |-----|
        | 0     | …    | …    | …               | …   |

**3. Data Preparation (Persiapan Data)**

Tahap *Data Preparation* bertujuan untuk menyiapkan data time series numerik agar dapat digunakan secara efektif oleh algoritma *machine learning*. Dataset **DucksAndGeese** tidak menyediakan file audio mentah (misalnya WAV/MP3), melainkan sinyal audio yang telah direpresentasikan dalam bentuk **time series numerik satu dimensi** dengan panjang yang seragam. Oleh karena itu, seluruh proses *preprocessing* dan *feature extraction* disesuaikan dengan karakteristik data tersebut.

**3.1 Karakteristik Input Data**

Setiap instance data memiliki karakteristik sebagai berikut:

* Bentuk data: *univariate time series* numerik
* Panjang sinyal: 236.784 titik waktu
* Jumlah kelas: 5 spesies burung (balanced)
* Tidak tersedia audio mentah atau metadata perekaman tambahan

Dengan kondisi ini, pendekatan yang paling rasional adalah mengekstraksi fitur numerik yang ringkas namun informatif untuk merepresentasikan karakteristik suara burung.

**3.2 Feature Extraction**

Karena penggunaan seluruh sinyal time series secara langsung tidak efisien untuk algoritma *machine learning* klasik, dilakukan proses ekstraksi fitur untuk mereduksi dimensi data sekaligus mempertahankan informasi penting.

Fitur yang diekstraksi terdiri dari 16 fitur numerik sebagai berikut:

1. **MFCC (Mel-frequency Cepstral Coefficients)**

   * Jumlah koefisien: 13
   * Merepresentasikan karakteristik spektral utama suara burung

2. **Zero Crossing Rate (ZCR)**

   * Mengukur frekuensi perubahan tanda sinyal
   * Mencerminkan karakteristik temporal suara

3. **RMS Energy**

   * Mengukur energi rata-rata sinyal
   * Merepresentasikan intensitas suara

4. **Spectral Centroid**

   * Menunjukkan pusat massa spektrum frekuensi
   * Berkaitan dengan tingkat "kecerahan" suara

Hasil ekstraksi fitur disusun dalam bentuk matriks fitur tabular dengan struktur sebagai berikut:

| mfcc1 | mfcc2 | ... | mfcc13 | ZCR | RMS | Centroid | label |
| ----- | ----- | --- | ------ | --- | --- | -------- | ----- |
3.3 Feature Normalization

Untuk menyamakan skala antar fitur numerik dan meningkatkan stabilitas proses pelatihan model, dilakukan normalisasi menggunakan *z-score normalization*:

[ X_{norm} = \frac{X - \mu}{\sigma} ]

Normalisasi dilakukan pada **fitur hasil ekstraksi**, bukan pada sinyal mentah, dengan parameter (\mu) dan (\sigma) dihitung dari data latih. Proses ini bertujuan untuk:

* Menghindari dominasi fitur tertentu
* Mempercepat konvergensi model
* Meningkatkan performa model berbasis jarak
3.4 Data Splitting

Dataset dibagi menjadi beberapa subset sebagai berikut:

1. Dataset asli:

   * Train set: 50 instance
   * Test set: 50 instance

2. Train set dibagi ulang menjadi:

   * 80% data latih
   * 20% data validasi

Data validasi digunakan untuk *hyperparameter tuning* dan pemilihan model terbaik, sedangkan data test hanya digunakan sekali pada evaluasi akhir untuk memastikan performa model yang objektif.



**4. Modeling (Pemodelan)**

Tahap *Modeling* bertujuan untuk membangun dan mengevaluasi model klasifikasi spesies burung berdasarkan fitur numerik hasil ekstraksi. Pendekatan utama yang digunakan adalah *feature-based machine learning*, karena paling sesuai dengan ukuran dan bentuk dataset.

**4.1 Model yang Digunakan**

**4.1.1 Model Utama: Random Forest**

Random Forest dipilih sebagai model utama karena memiliki performa yang stabil pada dataset berukuran kecil dan mampu menangani hubungan non-linear antar fitur.

Alasan pemilihan Random Forest:

* Stabil untuk dataset kecil (100 instance)
* Tahan terhadap *noise*
* Risiko *overfitting* relatif rendah
* Tidak memerlukan asumsi linearitas

**4.1.2 Model Alternatif: XGBoost**

XGBoost digunakan sebagai model pembanding untuk mengevaluasi apakah pendekatan *boosting* mampu memberikan peningkatan performa dibandingkan pendekatan *bagging* pada Random Forest.

Keunggulan XGBoost:

* Mampu memodelkan hubungan kompleks antar fitur
* Performa tinggi pada data tabular
* Digunakan secara luas pada penelitian klasifikasi audio berbasis fitur

**4.1.3 Model Baseline: K-Nearest Neighbors (KNN)**

KNN digunakan sebagai *baseline classifier* untuk memberikan tolok ukur performa minimum.

Karakteristik KNN:

* Implementasi sederhana
* Sensitif terhadap skala fitur
* Cocok sebagai pembanding awal

**4.2 Pipeline Pemodelan Feature-Based**

Pipeline pemodelan utama pada penelitian ini adalah sebagai berikut:

**Time Series Numerik → Feature Extraction → Feature Normalization → Machine Learning Model → Prediksi Kelas**

Pipeline ini dipilih karena paling sesuai dengan fakta dataset dan terbukti efektif untuk dataset audio berukuran kecil.
4.3 Pendekatan Pemodelan Alternatif (Konseptual)

Sebagai pembanding konseptual dan pengembangan lanjutan, beberapa pendekatan lain dipertimbangkan:

1. **DTW + KNN**
   Digunakan sebagai baseline klasik pada klasifikasi *time series* berbasis kemiripan bentuk sinyal.

2. **CNN dengan Spectrogram**
   Menggunakan representasi STFT atau Log-Mel spectrogram sebagai input CNN untuk menangkap pola frekuensi.

3. **LSTM**
   Digunakan untuk memodelkan dependensi temporal jangka panjang, namun berisiko *overfitting* pada dataset kecil.

Pendekatan-pendekatan ini tidak digunakan sebagai pipeline utama.

**4.4 Output Model**

Output yang dihasilkan oleh model meliputi:

* Prediksi kelas spesies burung (5 kelas)
* Nilai probabilitas prediksi (*confidence probability*)

Hasil ini digunakan sebagai dasar evaluasi performa model serta analisis lebih lanjut dalam penelitian.

**5. Evaluation (Evaluasi)**

Tahap *Evaluation* bertujuan untuk menilai kinerja model *machine learning* dalam mengklasifikasikan lima spesies burung pada dataset **DucksAndGeese** berdasarkan fitur audio hasil ekstraksi. Evaluasi dilakukan secara objektif menggunakan *test set* resmi yang disediakan oleh dataset agar hasil yang diperoleh merefleksikan kemampuan generalisasi model terhadap data yang belum pernah dilihat sebelumnya.

Untuk menjaga validitas penelitian, *test set* **hanya digunakan pada tahap evaluasi akhir** dan tidak terlibat dalam proses pelatihan maupun *hyperparameter tuning*, sehingga risiko *data leakage* dapat dihindari.

**5.1 Metrik Evaluasi**

Metrik evaluasi yang digunakan disesuaikan dengan karakteristik permasalahan klasifikasi multikelas dan ukuran dataset yang relatif kecil.

1. **Accuracy**
   Mengukur proporsi prediksi yang benar terhadap seluruh data uji.
   *Accuracy* digunakan sebagai indikator performa global model, dengan target performa **di atas baseline acak (20%)** dan *majority-class baseline*.

2. **Classification Report**
   Classification report digunakan untuk mengevaluasi performa model secara lebih rinci pada setiap kelas, yang mencakup:

   * **Precision**: ketepatan prediksi model untuk suatu kelas
   * **Recall**: kemampuan model mengenali instance dari kelas yang benar
   * **F1-score**: harmonisasi antara precision dan recall

   F1-score menjadi metrik utama karena memberikan gambaran yang seimbang terhadap kesalahan *false positive* dan *false negative* pada tiap spesies burung.

3. **Confusion Matrix**
   Confusion matrix digunakan untuk:

   * Melihat distribusi prediksi benar dan salah
   * Mengidentifikasi pasangan kelas yang sering tertukar
   * Menganalisis pola kesalahan klasifikasi antar spesies burung

**5.2 Prosedur Evaluasi Model**

Proses evaluasi model dilakukan secara bertahap sebagai berikut:

1. **Pelatihan Model**
   Data latih (*training set*) sebanyak 50 instance dibagi ulang menjadi:

   * 80% data latih
   * 20% data validasi

2. **Validasi Model**
   Data validasi digunakan untuk:

   * *Hyperparameter tuning* Random Forest (misalnya jumlah pohon dan kedalaman maksimum)
   * *Hyperparameter tuning* XGBoost (misalnya *learning rate* dan *max depth*)
   * Pemilihan model terbaik berdasarkan performa validasi

3. **Pengujian Akhir**

   * Model terbaik diuji menggunakan *test set* resmi (50 instance)
   * *Test set* tidak digunakan dalam proses pelatihan maupun validasi
   * Hasil evaluasi pada tahap ini digunakan sebagai hasil akhir penelitian

Pendekatan ini memastikan bahwa proses evaluasi dilakukan secara adil, objektif, dan bebas dari kebocoran data.

**5.3 Analisis dan Insight Evaluasi**

Berdasarkan hasil evaluasi model, dilakukan beberapa analisis untuk memperoleh pemahaman yang lebih mendalam terhadap perilaku model:

1. **Analisis Perbedaan Kelas Duck dan Goose**
   Evaluasi dilakukan untuk melihat apakah model lebih mudah membedakan kelompok *duck* dan *goose*. Secara umum, suara *duck* (khususnya *whistling duck*) memiliki karakteristik frekuensi yang lebih tinggi dibandingkan suara *goose*, sehingga diharapkan memiliki tingkat *recall* yang lebih baik.

2. **Analisis Kesalahan Klasifikasi**
   Confusion matrix digunakan untuk mengidentifikasi kelas yang paling sering tertukar, khususnya antar spesies *goose* yang memiliki pola frekuensi dan durasi suara yang relatif mirip.

3. **Faktor Penyebab Misclassification**
   Beberapa faktor yang diduga berkontribusi terhadap kesalahan klasifikasi meliputi:

   * Overlap karakteristik frekuensi antar spesies
   * Noise lingkungan pada rekaman audio
   * Variasi kualitas rekaman (kategori A dan B)

**Insight yang diperoleh dari tahap evaluasi ini digunakan untuk:**

* Menjelaskan keterbatasan model
* Memberikan justifikasi ilmiah terhadap hasil klasifikasi
* Menjadi dasar pengembangan model dan penelitian lanjutan

**6. Deployment (Streamlit)**

Tahap *Deployment* bertujuan untuk mengimplementasikan model klasifikasi yang telah dilatih ke dalam sebuah aplikasi interaktif berbasis web menggunakan **Streamlit**. Aplikasi ini memungkinkan pengguna melakukan inferensi model secara langsung tanpa harus memahami detail teknis *machine learning* yang mendasarinya.

Perancangan *deployment* disesuaikan dengan fakta dan batasan dataset **DucksAndGeese**, yaitu penggunaan *feature-based machine learning* sebagai *pipeline* utama. Oleh karena itu, seluruh proses inferensi dibuat **konsisten dengan tahap Data Preparation dan Modeling** untuk menghindari perbedaan distribusi data (*training–inference mismatch*).

**6.1 Arsitektur Deployment**

Aplikasi Streamlit berfungsi sebagai lapisan antarmuka (*front-end*) yang mengintegrasikan komponen berikut:

* *Feature extraction pipeline* (MFCC, ZCR, RMS Energy, Spectral Centroid)
* *Feature normalization* menggunakan parameter hasil pelatihan
* Inferensi model (*Random Forest / XGBoost*)
* Visualisasi hasil prediksi

Seluruh *pipeline preprocessing* yang digunakan pada aplikasi **identik dengan pipeline pada tahap training**, termasuk:

* Parameter ekstraksi fitur
* Urutan fitur
* Parameter normalisasi (*mean* dan *standard deviation*)

Model dan parameter *scaler* dimuat dari berkas hasil pelatihan (`.pkl`) sehingga tidak terjadi *data leakage* maupun inkonsistensi proses.

**6.2 Alur Kerja Aplikasi**

**6.2.1 Input Data**

Aplikasi mendukung dua skenario penggunaan:

1. **Input fitur numerik (utama)**
   Pengguna memasukkan atau memuat fitur audio hasil ekstraksi (MFCC, ZCR, RMS, Spectral Centroid) dalam format tabel.

2. **Input audio (opsional)**
   Audio `.wav` dapat diunggah untuk keperluan demonstrasi, kemudian diproses menggunakan *feature extraction pipeline* yang sama seperti pada tahap pelatihan.

Catatan penting:

* *Sample rate* diasumsikan **44.100 Hz**, sesuai dengan dataset DucksAndGeese
* Tidak dilakukan *downsampling* tambahan pada *pipeline* utama
* Audio disesuaikan panjangnya agar kompatibel dengan parameter ekstraksi fitur

**6.2.2 Proses Inferensi**

Tahapan inferensi model meliputi:

1. Ekstraksi fitur audio (jika input berupa audio)
2. Penyusunan vektor fitur berdimensi 16
3. Normalisasi fitur menggunakan *z-score normalization* berbasis data latih
4. Inferensi menggunakan model terlatih (Random Forest atau XGBoost)

Model menghasilkan:

* **Prediksi kelas** (1 dari 5 spesies burung)
* **Probabilitas prediksi** (*confidence score*) menggunakan `predict_proba`

**6.2.3 Output dan Visualisasi**

Aplikasi menampilkan beberapa komponen utama:

* Hasil prediksi spesies burung
* Grafik *confidence probability* untuk setiap kelas
* Visualisasi waveform audio (jika input berupa audio)
* Visualisasi *spectrogram* atau MFCC sebagai representasi domain frekuensi

Visualisasi ini bertujuan untuk meningkatkan interpretabilitas hasil prediksi dan memberikan pemahaman intuitif kepada pengguna.

**6.3 Struktur Navigasi Aplikasi**

Aplikasi Streamlit dibagi ke dalam beberapa halaman:

1. **Home**

   * Deskripsi singkat proyek
   * Tujuan klasifikasi suara burung
   * Ringkasan pendekatan *machine learning* yang digunakan

2. **EDA**

   * Contoh visualisasi waveform dan fitur audio
   * Perbandingan karakteristik suara antar spesies

3. **Prediction**

   * Input fitur atau audio
   * Proses inferensi model
   * Tampilan hasil prediksi dan *confidence score*

4. **About Dataset**

   * Informasi dataset DucksAndGeese
   * Jumlah kelas dan distribusi data
   * Karakteristik umum sinyal audio

**6.4 Struktur Berkas Deployment**

Struktur berkas aplikasi *deployment* adalah sebagai berikut:

```
streamlit_app.py          # File utama aplikasi Streamlit
model.pkl                 # Model terlatih (Random Forest / XGBoost)
scaler.pkl                # Parameter normalisasi fitur
feature_extraction.py     # Modul ekstraksi fitur audio
utils.py                  # Fungsi bantu visualisasi dan preprocessing
```

Struktur modular ini memudahkan pemeliharaan aplikasi serta memastikan konsistensi antara tahap pelatihan dan inferensi.

**6.5 Manfaat Deployment**

Implementasi *deployment* memberikan beberapa manfaat utama:

* Menyediakan bukti implementasi *end-to-end pipeline* penelitian
* Mempermudah validasi model secara praktis
* Menjembatani hasil penelitian dengan aplikasi nyata
* Memungkinkan pengujian data baru di luar dataset penelitian

Dengan adanya tahap *deployment*, penelitian ini tidak hanya bersifat teoritis tetapi juga aplikatif dan siap dikembangkan lebih lanjut.
